In [1]:
import cv2
import csv

In [2]:
cap = cv2.VideoCapture(r"C:\Users\MUHAMMADAHMAD\Desktop\InternShip Week 3\Test_Motion.mp4") 
ret, prev_frame = cap.read()
if not ret:
    print("Failed to read the video file.")
    cap.release()
    exit()
prev_frame1 = cv2.resize(prev_frame, (960, 540))
prev_gray = cv2.cvtColor(prev_frame1, cv2.COLOR_BGR2GRAY)
prev_blur = cv2.GaussianBlur(prev_gray, (25, 25), 0)

In [3]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
csv_file = open("motion_log.csv", mode="w", newline="")
csv_writer = csv.writer(csv_file)
csv_writer.writerow(["Frame", "Motion_Regions_Detected"])
frame_count = 0

In [4]:
DIFF_THRESHOLD = 10
MIN_CONTOUR_AREA = 500

In [5]:
while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break
    # 1. Convert to grayscale + Gaussian blur
    frame = cv2.resize(frame, (960, 540))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (25, 25), 0)

    # 2. Frame differencing
    diff = cv2.absdiff(prev_blur, blur)

    # 3. Threshold -> bi-level motion mask
    thresh = cv2.threshold(diff, DIFF_THRESHOLD, 255, cv2.THRESH_BINARY)[1]

    # 4. Morphological cleanup: opening (remove noise) -> dilation (merge fragments)
    # thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    thresh = cv2.dilate(thresh, kernel, iterations=2)

    # 5. Contour extraction
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 6. Draw bounding boxes for contours above the noise-filtering threshold
    motion_count = 0
    for c in contours:
        if cv2.contourArea(c) > MIN_CONTOUR_AREA:
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            motion_count += 1

    if motion_count > 0:
        cv2.putText(frame, "Motion Detected", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Log to CSV
    frame_count += 1
    csv_writer.writerow([frame_count, motion_count])

    # Display
    cv2.imshow("Motion Detection", frame)
    cv2.imshow("Motion Mask", thresh)

    # Update previous frame
    prev_blur = blur

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
csv_file.close()

print(f"Done. Processed {frame_count} frames. Log saved to motion_log.csv")

Done. Processed 580 frames. Log saved to motion_log.csv
